In [1]:
!nvidia-smi

[HAMI-core Msg(368:140410801006400:libvgpu.c:839)]: Initializing.....
Fri Aug 14 14:30:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:45:00.0 Off |                    0 |
| N/A   28C    P8             35W /  350W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |  

In [1]:
import gc
import json
import random
import time
from pathlib import Path

import pandas as pd
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import PeftModel

[HAMI-core Msg(176:140367156518208:libvgpu.c:839)]: Initializing.....


In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

from pathlib import Path
import random
import torch
import pandas as pd

PROJECT_DIR = Path(
    "/home/jovyan/project work/data_analyssis/fine tuning"
)

BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

ADAPTER_DIR = (
    PROJECT_DIR
    / "outputs"
    / "mistral_qlora"
    / "final_adapter"
)

TRAIN_DATA_PATH = (
    PROJECT_DIR
    / "outputs"
    / "cleaned_ruhsold_train.csv"
)

# ============================================================
# SYNTHETIC GENERATION OUTPUT DIRECTORIES
# ============================================================

GENERATION_DIR = (
    PROJECT_DIR
    / "outputs"
    / "synthetic_generation"
)

# Small pilot generations used to test prompting and QC
PILOT_OUTPUT_DIR = (
    GENERATION_DIR
    / "pilot"
)

# Final production generations used for augmentation
FULL_OUTPUT_DIR = (
    GENERATION_DIR
    / "full"
)

PILOT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FULL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# FINAL PRODUCTION SETTINGS
# ============================================================

PRODUCTION_BATCH_SIZE = 200

# ============================================================
# VERIFY PATHS
# ============================================================

print("Project directory:", PROJECT_DIR)
print("Adapter exists:", ADAPTER_DIR.exists())
print("Training data exists:", TRAIN_DATA_PATH.exists())
print("Pilot output:", PILOT_OUTPUT_DIR)
print("Full production output:", FULL_OUTPUT_DIR)
print("Production batch size:", PRODUCTION_BATCH_SIZE)

Project directory: /home/jovyan/project work/data_analyssis/fine tuning
Adapter exists: True
Training data exists: True
Pilot output: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot
Full production output: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full
Production batch size: 200


In [3]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)


# ============================================================
# FINAL 1x AUGMENTATION CONFIGURATION
# ============================================================

AUGMENTATION_CLASSES = [2, 3, 4]

CLASS_FILE_NAMES = {
    2: "religious_hate",
    3: "sexism",
    4: "profane",
}

# Required number of accepted synthetic examples
# after the complete QC pipeline.
ACCEPTED_TARGETS = {
    2: 500,
    3: 537,
    4: 411,
}

# Four authentic same-class demonstrations
# are used for every generation.
N_DEMONSTRATIONS = 4

# Deterministic seed bases.
BASE_GENERATION_SEED = 1000
BASE_DEMONSTRATION_SEED = 5000

print("Final 1x accepted synthetic targets:")

for class_id, target in ACCEPTED_TARGETS.items():
    print(
        f"Class {class_id}: {target}"
    )

print(
    "\nTotal accepted target:",
    sum(ACCEPTED_TARGETS.values())
)

Final 1x accepted synthetic targets:
Class 2: 500
Class 3: 537
Class 4: 411

Total accepted target: 1448


[HAMI-core Msg(176:140367156518208:libvgpu.c:855)]: Initialized


In [4]:
train_df = pd.read_csv(TRAIN_DATA_PATH)

print("Training data shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

train_df.head()

Training data shape: (6401, 2)
Columns: ['text', 'label']


,text,label
0,kia howa hai aap ko allah bless and protect yo...,1
1,randdi hai,3
2,smjh to agai thi mjhy,1
3,haan yrr tuny sahi thukayi ki abhi tak lund da...,3
4,rundi ka bacha bharwaaa ptm ka kuttaa,0


In [5]:
# ============================================================
# PREPARE AUTHENTIC DEMONSTRATION POOL
# ============================================================

def prepare_demonstration_pool(
    dataframe,
    text_column="text",
    label_column="label",
):
    required_columns = {
        text_column,
        label_column,
    }

    missing_columns = (
        required_columns.difference(
            dataframe.columns
        )
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    clean_df = dataframe[
        [text_column, label_column]
    ].copy()

    clean_df[text_column] = (
        clean_df[text_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    clean_df[label_column] = (
        pd.to_numeric(
            clean_df[label_column],
            errors="coerce",
        )
    )

    clean_df = clean_df[
        clean_df[text_column].ne("")
        &
        clean_df[label_column].notna()
    ].copy()

    clean_df[label_column] = (
        clean_df[label_column]
        .astype(int)
    )

    # One copy of each authentic text per class.
    clean_df = (
        clean_df
        .drop_duplicates(
            subset=[
                text_column,
                label_column,
            ]
        )
        .reset_index(drop=True)
    )

    return clean_df


demonstration_pool = (
    prepare_demonstration_pool(
        train_df
    )
)

print(
    "Clean demonstration pool:",
    demonstration_pool.shape
)

print("\nClass distribution:")

print(
    demonstration_pool["label"]
    .value_counts()
    .sort_index()
)

Clean demonstration pool: (6401, 2)

Class distribution:
label
0    1534
1    3422
2     500
3     535
4     410
Name: count, dtype: int64


In [7]:
# ============================================================
# Display original RUHSOLD Sexism examples
# ============================================================

sexism_examples = (
    demonstration_pool[
        demonstration_pool["label"] == 3
    ][["text", "label"]]
    .drop_duplicates(subset=["text"])
    .sample(
        n=10,
        random_state=42
    )
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", None)

for i, row in sexism_examples.iterrows():
    print("=" * 80)
    print(f"Example {i+1}")
    print("=" * 80)
    print(row["text"])
    print()

Example 1
gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,

Example 2
yar is pagal ko dafa karo islam ko badnam kr rahi ha randi kahin ki.ak video men nangi ho jati or agli video men hajan ban jati ha

Example 3
sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho

Example 4
bc nasal ki randi

Example 5
to be na randi hy na asle....shame on u

Example 6
abu se bhai jaan se chacha jaan sabhi se to halala krwa leti hai begum  or ye tera jeeja ravish khan usse bhi  salon humse akele he kar denge

Example 7
#rashamidesai tu randi moti apni maa ko bech chuki hai ...kisi aur ke hatho..to apni awkat ne rah kar game plan kar ...arhan to vaduaa hai...ram mandir ban rha hai na..but tu chud rahi hai kisi aur ka bister garam kar rhi hai

Example 8
randi tere khene se koch nahi hoga kanjar nasal ki payedawar

Example 9
yeh dekh chakki union ki randi

Example 10
zartaj gul tu mera lora chup ghastiyan lully khanay wali ghashti imran niazi

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer loaded
Pad token: </s>
EOS token: </s>


In [9]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit quantization configured")

4-bit quantization configured


In [10]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

base_model.config.use_cache = True

print("Base model loaded")
print("Model device:", next(base_model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded
Model device: cuda:0


In [11]:
finetuned_model = PeftModel.from_pretrained( base_model, ADAPTER_DIR, is_trainable=False, )
finetuned_model.eval()
print("Fine-tuned model loaded")
print("Active adapter:", finetuned_model.active_adapter)

Fine-tuned model loaded
Active adapter: default


In [12]:
# ============================================================
# FINAL PRODUCTION DEMONSTRATION SAMPLER
# ============================================================

def sample_demonstrations(
    dataframe,
    class_id,
    n_examples=4,
    random_state=42,
    text_column="text",
    label_column="label",
):
    """
    Sample unique authentic RUHSOLD demonstrations
    from the requested target class.

    A deterministic random seed is used so that the
    demonstration selection is reproducible.
    """

    # Select authentic examples belonging to the target class.
    class_pool = (
        dataframe[
            dataframe[label_column] == class_id
        ][text_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Remove empty and duplicate texts.
    class_pool = (
        class_pool[
            class_pool.ne("")
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(class_pool) < n_examples:
        raise ValueError(
            f"Class {class_id} contains only "
            f"{len(class_pool)} unique usable examples, "
            f"but {n_examples} were requested."
        )

    demonstrations = (
        class_pool
        .sample(
            n=n_examples,
            replace=False,
            random_state=random_state,
        )
        .tolist()
    )

    # Final safety check.
    if len(demonstrations) != len(set(demonstrations)):
        raise RuntimeError(
            "Duplicate demonstrations detected."
        )

    return demonstrations

In [13]:
# ============================================================
# RUHSOLD CLASS METADATA
# ============================================================

PROMPT_VERSION = "v3_original_demo_boundary_prompt"

CLASS_METADATA = {
    0: {
        "label": "Abusive/Offensive",

        "definition": (
            "Profane, strongly impolite, rude, or vulgar language used to insult "
            "or hurt a targeted individual or group."
        ),

        "boundary": (
            "The language must have a clear target. If the attack is mainly based "
            "on religion or gender, it belongs to Religious Hate or Sexism. "
            "Vulgar language without an intended target belongs to Profane."
        ),
    },

    1: {
        "label": "Normal",

        "definition": (
            "Text that does not belong to the Abusive/Offensive, Sexism, "
            "Religious Hate, or Profane categories."
        ),

        "boundary": (
            "The post must not contain targeted abuse, gender-based hatred, "
            "religious hatred, or vulgar language."
        ),
    },

    2: {
        "label": "Religious Hate",

        "definition": (
            "Language expressing hatred towards a targeted individual or group "
            "because of religious beliefs, lack of religious beliefs, or religious "
            "identity. It may also use religion to encourage hatred or violence."
        ),

        "boundary": (
            "Religion must be the main reason for the hostility. General political "
            "criticism, personal abuse, or profanity without a clear religious "
            "motivation is not Religious Hate."
        ),
    },

    3: {
        "label": "Sexism",

        "definition": (
            "Language expressing hatred towards a targeted individual or group "
            "because of gender or sexual orientation."
        ),

        "boundary": (
            "Gender or sexual orientation must be the main reason for the hostility. "
            "General insults, sexual vulgarity, or references to female relatives "
            "without gender-based hatred are not Sexism."
        ),
    },

    4: {
        "label": "Profane",

        "definition": (
            "The use of vulgar, foul, or obscene language without an intended target."
        ),

        "boundary": (
            "The profanity should not be directed at a specific person or group. "
            "Targeted insults belong to Abusive/Offensive, while religion-based "
            "or gender-based attacks belong to their respective classes."
        ),
    },
}

print("Prompt version:", PROMPT_VERSION)
print("Class metadata loaded successfully.\n")

for class_id, class_info in CLASS_METADATA.items():
    print(
        f"{class_id}: "
        f"{class_info['label']}"
    )

Prompt version: v3_original_demo_boundary_prompt
Class metadata loaded successfully.

0: Abusive/Offensive
1: Normal
2: Religious Hate
3: Sexism
4: Profane


In [14]:
# ============================================================
# BUILD FINAL FEW-SHOT GENERATION PROMPT
# ============================================================

def build_few_shot_messages(
    class_id,
    demonstrations,
):
    """
    Build the production few-shot prompt for one RUHSOLD class.
    """

    if class_id not in CLASS_METADATA:
        raise ValueError(
            f"Unknown class ID: {class_id}. "
            f"Valid IDs: {list(CLASS_METADATA.keys())}"
        )

    class_info = CLASS_METADATA[class_id]

    # --------------------------------------------------------
    # Clean demonstrations
    # --------------------------------------------------------

    demonstration_texts = [
        str(text).strip()
        for text in demonstrations
        if pd.notna(text)
        and str(text).strip()
    ]

    if len(demonstration_texts) != N_DEMONSTRATIONS:
        raise ValueError(
            f"Expected {N_DEMONSTRATIONS} demonstrations, "
            f"received {len(demonstration_texts)}."
        )

    if len(demonstration_texts) != len(
        set(demonstration_texts)
    ):
        raise ValueError(
            "Duplicate demonstrations were supplied "
            "to the prompt builder."
        )

    # --------------------------------------------------------
    # Format examples
    # --------------------------------------------------------

    formatted_examples = "\n".join(
        f"Example {index}: {text}"
        for index, text in enumerate(
            demonstration_texts,
            start=1,
        )
    )

    # --------------------------------------------------------
    # Class-specific generation requirement
    # --------------------------------------------------------

    if class_id == 2:

        class_specific_requirement = """
- The generated post MUST clearly express hostility toward a person
  or group because of their religion, religious belief, or religious identity.
- Simply using profanity or insulting somebody is NOT sufficient.
- Simply mentioning a religion or religious identity is NOT sufficient.
- The religious identity must be directly connected to the hostility.
""".strip()

    elif class_id == 3:

        class_specific_requirement = """
- The generated post MUST clearly express sexism, gender-based hostility,
  gender stereotyping, or degradation based on gender or sexual orientation.
- Generic profanity or a generic personal insult is NOT sufficient.
- Sexual or vulgar language alone is NOT sufficient.
- Gender or sexual identity must be directly connected to the hostility,
  stereotype, or degradation.
""".strip()

    elif class_id == 4:

        class_specific_requirement = """
- The generated post MUST contain profanity used as a general expression,
  exclamation, intensifier, frustration, or vulgar remark.
- The profanity must NOT primarily attack a specific person or group.
- Do NOT generate religious hate or gender-based hostility.
- Do NOT turn the post into a targeted personal insult.
""".strip()

    else:

        class_specific_requirement = """
- The meaning of the post must clearly match the target class.
""".strip()

    # --------------------------------------------------------
    # Final production prompt
    # --------------------------------------------------------

    instruction = f"""
You are generating synthetic Roman Urdu social-media posts for an
academic hate-speech classification dataset.

Target class: {class_info["label"]}

Definition:
{class_info["definition"]}

Important class boundary:
{class_info["boundary"]}

Real examples from the target class:

{formatted_examples}

Generate exactly one new, natural and informal Roman Urdu social-media
post that clearly belongs to the target class.

Requirements:
- Write primarily in Roman Urdu using the Latin alphabet.
- Natural English code-mixing is allowed.
{class_specific_requirement}
- Do not copy or closely paraphrase the examples.
- Do not include a label, explanation, translation, or quotation marks.
- Return only the generated post.
""".strip()

    return [
        {
            "role": "user",
            "content": instruction,
        }
    ]

In [15]:
# ============================================================
# TEST FINAL PROMPT
# ============================================================

test_demonstrations = sample_demonstrations(
    dataframe=demonstration_pool,
    class_id=3,
    n_examples=N_DEMONSTRATIONS,
    random_state=42,
)

test_messages = build_few_shot_messages(
    class_id=3,
    demonstrations=test_demonstrations,
)

print(test_messages[0]["content"])

You are generating synthetic Roman Urdu social-media posts for an
academic hate-speech classification dataset.

Target class: Sexism

Definition:
Language expressing hatred towards a targeted individual or group because of gender or sexual orientation.

Important class boundary:
Gender or sexual orientation must be the main reason for the hostility. General insults, sexual vulgarity, or references to female relatives without gender-based hatred are not Sexism.

Real examples from the target class:

Example 1: gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,
Example 2: yar is pagal ko dafa karo islam ko badnam kr rahi ha randi kahin ki.ak video men nangi ho jati or agli video men hajan ban jati ha
Example 3: sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho
Example 4: bc nasal ki randi

Generate exactly one new, natural and informal Roman Urdu social-media
post that clearly belongs to the target class.

Requirements:
- Wr

In [16]:

GENERATION_CONFIG = {
    "max_new_tokens": 60,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.90,
    "repetition_penalty": 1.15,
}

In [17]:
@torch.inference_mode()
def generate_one(
    model,
    messages,
    seed,
    generation_config=None,
):
    if generation_config is None:
        generation_config = GENERATION_CONFIG

    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value in model_inputs.items()
    }

    output_ids = model.generate(
        **model_inputs,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        **generation_config,
    )

    prompt_length = (
        model_inputs["input_ids"]
        .shape[1]
    )

    generated_ids = output_ids[
        0,
        prompt_length:
    ]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated_text

In [33]:
RELIGIOUS_PROTOTYPE_KEEP_IDS = [
    2, 3, 5, 7, 8, 10, 11, 13, 18, 23,
    33, 34, 37, 38, 41, 48, 50, 57, 60, 62,
    63, 65, 68, 71, 74, 75, 76, 78, 86, 92,
    94, 98, 99, 100, 107, 109, 118, 124, 125, 131,
    136, 138, 140, 150, 153, 155, 157, 160, 161, 162,
    163, 165, 180, 182, 189, 193, 198, 200, 209, 220,
    222, 224, 227, 229, 232, 237, 238, 243, 247, 254,
    255, 262, 268, 269, 270, 276, 279, 292, 296, 308,
    313, 315, 320, 327, 330, 336, 338, 340, 343, 345,
    347, 348, 350, 358, 359, 376, 384, 387, 389, 391,
    398, 401, 405, 412, 427, 430, 432, 435, 439, 440,
    449, 452, 454, 457, 468, 486, 494, 498
]

In [34]:
# ============================================================
# PREVIEW FINAL PRODUCTION GENERATIONS
# ============================================================

def preview_generations(
    class_id,
    n_samples=5,
    start_index=0,
    show_demonstrations=False,
):
    """
    Generate a small preview batch using the final
    production prompting strategy.
    """

    if class_id not in CLASS_METADATA:
        raise ValueError(
            f"Unknown class ID: {class_id}"
        )

    # --------------------------------------------------------
    # Select demonstration source
    # --------------------------------------------------------

    if class_id == 2:

        demo_source_df = religious_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Religious Hate examples"
        )

    elif class_id == 3:

        demo_source_df = sexism_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Sexism examples"
        )

    elif class_id == 4:

        demo_source_df = profane_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Profane examples"
        )

    else:

        demo_source_df = demonstration_pool

        demo_source_name = (
            "Random authentic same-class RUHSOLD examples"
        )

    # --------------------------------------------------------
    # Display generation setup
    # --------------------------------------------------------

    print("=" * 80)

    print(
        "Target class:",
        CLASS_METADATA[class_id]["label"]
    )

    print(
        "Demonstration source:",
        demo_source_name
    )

    print(
        "Demonstration pool size:",
        len(demo_source_df)
    )

    print("=" * 80)

    results = []

    # --------------------------------------------------------
    # Generate preview samples
    # --------------------------------------------------------

    for offset in range(n_samples):

        sample_index = (
            start_index + offset
        )

        # ----------------------------------------------------
        # Reproducible seeds
        # ----------------------------------------------------

        generation_seed = (
            BASE_GENERATION_SEED
            + class_id * 100000
            + sample_index
        )

        demonstration_seed = (
            BASE_DEMONSTRATION_SEED
            + class_id * 100000
            + sample_index
        )

        # ----------------------------------------------------
        # Sample four unique authentic demonstrations
        # ----------------------------------------------------

        demonstrations = sample_demonstrations(
            dataframe=demo_source_df,
            class_id=class_id,
            n_examples=N_DEMONSTRATIONS,
            random_state=demonstration_seed,
        )

        # ----------------------------------------------------
        # Build prompt
        # ----------------------------------------------------

        messages = build_few_shot_messages(
            class_id=class_id,
            demonstrations=demonstrations,
        )

        # ----------------------------------------------------
        # Generate
        # ----------------------------------------------------

        generated_text = generate_one(
            model=finetuned_model,
            messages=messages,
            seed=generation_seed,
        )

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        print(
            f"\nSample {offset + 1}"
        )

        print("-" * 60)

        print(
            "Sample index:",
            sample_index
        )

        print(
            "Generation seed:",
            generation_seed
        )

        print(
            "Demonstration seed:",
            demonstration_seed
        )

        if show_demonstrations:

            print("\nDemonstrations:")

            for index, example in enumerate(
                demonstrations,
                start=1,
            ):
                print(
                    f"{index}. {example}"
                )

        print("\nGenerated output:")
        print(generated_text)

        # ----------------------------------------------------
        # Record result
        # ----------------------------------------------------

        results.append({
            "sample_index":
                sample_index,

            "class_id":
                class_id,

            "target_label":
                CLASS_METADATA[
                    class_id
                ]["label"],

            "generation_seed":
                generation_seed,

            "demonstration_seed":
                demonstration_seed,

            "prompt_version":
                PROMPT_VERSION,

            "prompting_strategy":
                "4-shot random authentic same-class",

            "demonstration_source":
                demo_source_name,

            "demonstration_pool_size":
                len(demo_source_df),

            "demonstrations":
                demonstrations,

            "generated_text":
                generated_text,
        })

    return results

In [40]:
# ============================================================
# CHECK AVAILABLE RELIGIOUS-HATE DATAFRAMES
# ============================================================

for name, obj in globals().items():

    if (
        isinstance(obj, pd.DataFrame)
        and "relig" in name.lower()
    ):
        print(
            name,
            obj.shape,
            obj.columns.tolist()
        )

In [41]:
# ============================================================
# CHECK ALL DATAFRAMES CURRENTLY IN MEMORY
# ============================================================

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        print(
            f"{name}: "
            f"shape={obj.shape}"
        )

        print(
            "Columns:",
            obj.columns.tolist()
        )

        print("-" * 80)

_: shape=(5, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
train_df: shape=(6401, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
_4: shape=(5, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
demonstration_pool: shape=(6401, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
sexism_examples: shape=(10, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
sexism_candidates: shape=(535, 3)
Columns: ['review_id', 'text', 'label']
--------------------------------------------------------------------------------
sexism_prototype_pool: shape=(40, 2)
Columns: ['text', 'label']
--------------------------------------------------------------------------------
profane_candidates: shape=(410, 2)

In [42]:
# ============================================================
# RECONSTRUCT RELIGIOUS HATE PROTOTYPE POOL
# ============================================================

# First isolate authentic Religious Hate examples
religious_candidates = (
    demonstration_pool[
        demonstration_pool["label"] == 2
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Religious Hate authentic candidates:",
    len(religious_candidates)
)

# ------------------------------------------------------------
# Rebuild curated prototype pool from saved keep IDs
# ------------------------------------------------------------

religious_prototype_pool = (
    religious_candidates
    .iloc[
        RELIGIOUS_PROTOTYPE_KEEP_IDS
    ]
    [
        [
            "text",
            "label",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Religious prototype pool size:",
    len(religious_prototype_pool)
)

print(
    "\nClass distribution:"
)

print(
    religious_prototype_pool[
        "label"
    ].value_counts()
)

display(
    religious_prototype_pool.head(10)
)

Religious Hate authentic candidates: 500
Religious prototype pool size: 118

Class distribution:
label
2    118
Name: count, dtype: int64


,text,label
0,"insaniyat ko bache rahne do, jis mazhab me janwar ka gala ret ke khushi banti hai, jis mazhab me soorma logo ko 72 virgin ladkiya baati jaati, jo duniya pe kabze ke sapne dekhta ho us mazhab se duuri banane me hi insaniyat ki bhalayi hai.",2
1,1 balatkari muslim 3 balatkari tera baap hindu h uske bare m kuch bol be,2
2,hindu dharm ka matlab samajh pehle puppet khali bhaunk aur bhaunk iam enjoying ur bark 😂😂😂😂,2
3,jahil k bachy kahan likha h k gair mazhb sy nikaah jaiz ha... tmari gand me jo keera ha wo nhi merna... khoty khany waly... yahoodi kahan sy hogaey ahle kitaab ....tm.b yahoodi to nhi,2
4,lanat abbtknews k khilaf trend bnane walye yazeedion pe r bibi k dushmano p bhi,2
5,teri maa ka halala kra rha tha koi problem madarjat bhosdike mulle,2
6,apne profile mein whats app me fb twitter mein is bahadur nojawaan ki pic lga kr islam se or nabi ke din se mhbt ka izhaar kren or itna share kren ek yahoodi apne na pak iradoon se baaz ajaye is shakhs ne quraan pak ki hifazahat mein ek bhtareen mhbt ka jita jagta sabot dia hai,2
7,yahoodi ko hitler ne chora or general ne in jaajoon ko chor k dhkya k ye kai hain.😂,2
8,lul bukhari ye tum nahe tumaray yahoodi or hindu thuku bol rahay hain jin ko army sey bahot hi zeda takleef hai...🖕,2
9,pislam me media ka estemal kerna haram hai halala ki paidais,2


In [43]:
# ============================================================
# SANITY CHECK ALL CURATED DEMONSTRATION POOLS
# ============================================================

print(
    "Religious Hate:",
    len(religious_prototype_pool),
    "samples"
)

print(
    "Sexism:",
    len(sexism_prototype_pool),
    "samples"
)

print(
    "Profane:",
    len(profane_prototype_pool),
    "samples"
)

assert set(
    religious_prototype_pool["label"].unique()
) == {2}

assert set(
    sexism_prototype_pool["label"].unique()
) == {3}

assert set(
    profane_prototype_pool["label"].unique()
) == {4}

print(
    "\nAll curated pools are valid."
)

Religious Hate: 118 samples
Sexism: 40 samples
Profane: 40 samples

All curated pools are valid.


In [44]:
# ============================================================
# SAVE CURATED DEMONSTRATION POOLS
# ============================================================

CURATED_POOL_DIR = (
    GENERATION_DIR
    / "curated_demonstration_pools"
)

CURATED_POOL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

religious_prototype_pool.to_csv(
    CURATED_POOL_DIR
    / "religious_hate_prototype_pool.csv",
    index=False,
    encoding="utf-8"
)

sexism_prototype_pool.to_csv(
    CURATED_POOL_DIR
    / "sexism_prototype_pool.csv",
    index=False,
    encoding="utf-8"
)

profane_prototype_pool.to_csv(
    CURATED_POOL_DIR
    / "profane_prototype_pool.csv",
    index=False,
    encoding="utf-8"
)

print(
    "Curated demonstration pools saved."
)

Curated demonstration pools saved.


In [45]:
# ============================================================
# FINAL SANITY CHECK
# Religious Hate: 5 samples
# ============================================================

religious_test_samples = preview_generations(
    class_id=2,
    n_samples=5,
    start_index=100,
    show_demonstrations=True,
)

Target class: Religious Hate
Demonstration source: Shortlisted authentic RUHSOLD Religious Hate examples
Demonstration pool size: 118

Sample 1
------------------------------------------------------------
Sample index: 100
Generation seed: 201100
Demonstration seed: 205100

Demonstrations:
1. kaafir tujhe kuch nai malum
2. madarchod muslaman ki kaum madarchod saalo ka imaan nahi hai bheno ko chod te hai bhen ke lode harmazaade uskiye apni  maa ki chut bhi chodta hoga aur chuswata hoga logo se saare musalamaan raand ki olad islam ki maa ki chut
3. tum log yahoodi ho kisi ko shuck nahi hona chaheye bus kafi hogae musalmanon ko aur dhoka na dou👹
4. #stopisraeliterror   yahoodi mulk jisey israel ka naam diya gaya najaiz hai yahoodi sabse bada dahshatgard

Generated output:
abe kuttay ye mullah jo teri behn ko chod raha hai wo maulana hein

Sample 2
------------------------------------------------------------
Sample index: 101
Generation seed: 201101
Demonstration seed: 205101

Demonstratio

In [ ]:
# ============================================================
# PREVIEW WITH BASE MISTRAL
# ADAPTER DISABLED
# ============================================================

def preview_base_model_generations(
    class_id,
    n_samples=5,
    start_index=0,
    show_demonstrations=False,
):
    """
    Generate using the same prompts and seeds,
    but with the QLoRA adapter temporarily disabled.
    """

    if class_id not in CLASS_METADATA:
        raise ValueError(
            f"Unknown class ID: {class_id}"
        )

    # --------------------------------------------------------
    # Select demonstration source
    # --------------------------------------------------------

    if class_id == 2:
        demo_source_df = religious_prototype_pool
        demo_source_name = (
            "Shortlisted authentic RUHSOLD Religious Hate examples"
        )
    else:
        demo_source_df = demonstration_pool
        demo_source_name = (
            "Random authentic same-class RUHSOLD examples"
        )

    print("=" * 80)
    print(
        "MODEL: BASE MISTRAL (QLoRA ADAPTER DISABLED)"
    )
    print(
        "Target class:",
        CLASS_METADATA[class_id]["label"]
    )
    print(
        "Demonstration source:",
        demo_source_name
    )
    print("=" * 80)

    results = []

    # --------------------------------------------------------
    # Temporarily disable QLoRA adapter
    # --------------------------------------------------------

    with finetuned_model.disable_adapter():

        for offset in range(n_samples):

            sample_index = start_index + offset

            # ------------------------------------------------
            # Reproducible seeds
            # ------------------------------------------------

            generation_seed = (
                BASE_GENERATION_SEED
                + class_id * 100000
                + sample_index
            )

            demonstration_seed = (
                BASE_DEMONSTRATION_SEED
                + class_id * 100000
                + sample_index
            )

            # ------------------------------------------------
            # Sample demonstrations
            # ------------------------------------------------

            demonstrations = sample_demonstrations(
                dataframe=demo_source_df,
                class_id=class_id,
                n_examples=N_DEMONSTRATIONS,
                random_state=demonstration_seed,
            )

            # ------------------------------------------------
            # Build identical prompt
            # ------------------------------------------------

            messages = build_few_shot_messages(
                class_id=class_id,
                demonstrations=demonstrations,
            )

            # ------------------------------------------------
            # Generate
            # ------------------------------------------------

            generated_text = generate_one(
                model=finetuned_model,
                messages=messages,
                seed=generation_seed,
            )

            # ------------------------------------------------
            # Display
            # ------------------------------------------------

            print(
                f"\nSample {offset + 1}"
            )

            print("-" * 60)

            print(
                "Sample index:",
                sample_index
            )

            print(
                "Generation seed:",
                generation_seed
            )

            print(
                "Demonstration seed:",
                demonstration_seed
            )

            if show_demonstrations:

                print("\nDemonstrations:")

                for index, example in enumerate(
                    demonstrations,
                    start=1,
                ):
                    print(
                        f"{index}. {example}"
                    )

            print("\nGenerated output:")
            print(generated_text)

            # ------------------------------------------------
            # Save result
            # ------------------------------------------------

            results.append({
                "sample_index":
                    sample_index,

                "class_id":
                    class_id,

                "target_label":
                    CLASS_METADATA[
                        class_id
                    ]["label"],

                "generation_seed":
                    generation_seed,

                "demonstration_seed":
                    demonstration_seed,

                "model":
                    "Base Mistral-7B-Instruct-v0.2",

                "adapter":
                    "disabled",

                "demonstrations":
                    demonstrations,

                "generated_text":
                    generated_text,
            })

    return results

In [ ]:
religious_base_test_samples = preview_base_model_generations(
    class_id=2,
    n_samples=5,
    start_index=100,
    show_demonstrations=True,
)

In [ ]:
# ============================================================
# GENERATE PILOT DATASET
# ============================================================

def generate_pilot_dataset(
    class_id,
    n_samples=100,
    start_index=0,
    save_every=10,
):
    """
    Generate and save a reproducible pilot synthetic dataset
    for one RUHSOLD target class.

    Religious Hate uses the curated Religious Hate prototype pool.
    Sexism uses the curated Sexism prototype pool.
    Profane uses the curated Profane prototype pool.
    Other classes use the general authentic demonstration pool.
    """

    if class_id not in CLASS_METADATA:
        raise ValueError(
            f"Unknown class ID: {class_id}. "
            f"Valid IDs: {list(CLASS_METADATA.keys())}"
        )

    # --------------------------------------------------------
    # Select demonstration source
    # --------------------------------------------------------

    if class_id == 2:

        demo_source_df = religious_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Religious Hate examples"
        )

    elif class_id == 3:

        demo_source_df = sexism_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Sexism examples"
        )

    elif class_id == 4:

        demo_source_df = profane_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Profane examples"
        )

    else:

        demo_source_df = demonstration_pool

        demo_source_name = (
            "Random authentic same-class RUHSOLD examples"
        )

    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    PILOT_OUTPUT_DIR = (
        PROJECT_ROOT
        / "fine tuning"
        / "outputs"
        / "synthetic_generation"
        / "pilot"
    )

    PILOT_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Build output filename
    # --------------------------------------------------------

    class_name_for_file = (
        CLASS_METADATA[class_id]["label"]
        .lower()
        .replace("/", "_")
        .replace(" ", "_")
    )

    output_path = (
        PILOT_OUTPUT_DIR
        / f"{class_name_for_file}_raw_pilot_{n_samples}.csv"
    )

    # --------------------------------------------------------
    # Display setup
    # --------------------------------------------------------

    print("=" * 80)
    print("Generating pilot dataset")

    print(
        "Class:",
        CLASS_METADATA[class_id]["label"]
    )

    print(
        "Demonstration source:",
        demo_source_name
    )

    print(
        "Demonstration pool size:",
        len(demo_source_df)
    )

    print(
        "Samples:",
        n_samples
    )

    print(
        "Output:",
        output_path
    )

    print("=" * 80)

    # --------------------------------------------------------
    # Generate samples
    # --------------------------------------------------------

    results = []

    for offset in range(n_samples):

        sample_index = (
            start_index + offset
        )

        # ----------------------------------------------------
        # Reproducible seeds
        # ----------------------------------------------------

        generation_seed = (
            BASE_GENERATION_SEED
            + class_id * 100000
            + sample_index
        )

        demonstration_seed = (
            BASE_DEMONSTRATION_SEED
            + class_id * 100000
            + sample_index
        )

        # ----------------------------------------------------
        # Sample demonstrations
        # ----------------------------------------------------

        demonstrations = sample_demonstrations(
            dataframe=demo_source_df,
            class_id=class_id,
            n_examples=N_DEMONSTRATIONS,
            random_state=demonstration_seed,
        )

        # ----------------------------------------------------
        # Build generation prompt
        # ----------------------------------------------------

        messages = build_few_shot_messages(
            class_id=class_id,
            demonstrations=demonstrations,
        )

        # ----------------------------------------------------
        # Generate synthetic sample
        # ----------------------------------------------------

        generated_text = generate_one(
            model=finetuned_model,
            messages=messages,
            seed=generation_seed,
        )

        # ----------------------------------------------------
        # Store generation record
        # ----------------------------------------------------

        result = {
            "sample_index":
                sample_index,

            "class_id":
                class_id,

            "target_label":
                CLASS_METADATA[
                    class_id
                ]["label"],

            "generation_seed":
                generation_seed,

            "demonstration_seed":
                demonstration_seed,

            "prompt_version":
                PROMPT_VERSION,

            "prompting_strategy":
                "4-shot random authentic same-class",

            "demonstration_source":
                demo_source_name,

            "demonstration_pool_size":
                len(demo_source_df),

            "demo_1":
                demonstrations[0],

            "demo_2":
                demonstrations[1],

            "demo_3":
                demonstrations[2],

            "demo_4":
                demonstrations[3],

            "generated_text":
                generated_text,
        }

        results.append(result)

        # ----------------------------------------------------
        # Periodic checkpoint save
        # ----------------------------------------------------

        if (
            (offset + 1) % save_every == 0
            or
            (offset + 1) == n_samples
        ):

            temporary_df = pd.DataFrame(
                results
            )

            temporary_df.to_csv(
                output_path,
                index=False,
                encoding="utf-8"
            )

            print(
                f"Saved {offset + 1}/{n_samples}"
            )

    # --------------------------------------------------------
    # Final dataframe
    # --------------------------------------------------------

    pilot_df = pd.DataFrame(
        results
    )

    # --------------------------------------------------------
    # Final save
    # --------------------------------------------------------

    pilot_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    print("\nGeneration complete.")

    print(
        "Shape:",
        pilot_df.shape
    )

    print(
        "Saved to:",
        output_path
    )

    return pilot_df

In [54]:
religious_pilot_df = generate_pilot_dataset(
    class_id=2,
    n_samples=100,
    start_index=0,
    save_every=10,
)

Generating pilot dataset
Class: Religious Hate
Samples: 100
Output: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/religious_hate_raw_pilot_100.csv
Saved 10/100
Saved 20/100
Saved 30/100
Saved 40/100
Saved 50/100
Saved 60/100
Saved 70/100
Saved 80/100
Saved 90/100
Saved 100/100

Generation complete.
Shape: (100, 14)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/religious_hate_raw_pilot_100.csv


In [55]:
religious_pilot_df[
    [
        "sample_index",
        "generated_text",
    ]
].head(20)

,sample_index,generated_text
0,0,tery baap ne tu gaddar hi banaya tha...tum sab muslim hn or bhag jao
1,1,teri maa ki phudi mein lanti tere jaise muslamanon ke nikal
2,2,yar baat mat karte tu apny baap ka baap sy larkiyon ko bachana chahtay ho???
3,3,mulle bharwe yahan wahan aati hai ye duniya hamara hi mulk hai
4,4,"abey, behan tu samaj kro, mera jaan hai"
5,5,apni maa ko goli marne wala teri maa chod gayee thi saalee
6,6,ye logon ne baz e mohtaj ki zarurat hi nahi kaha thi usko phly tumhee yahan se pta chala ga
7,7,tere baap ke pass pakistan ki dushman hain yahodi
8,8,aurat march se harami halala ki pedawar
9,9,jb tu hmari soch sy mohsin na ho ga tab to esa hi karwa dunga koi


In [22]:
# ============================================================
# FINAL SANITY CHECK
# Sexism: 5 samples
# ============================================================

sexism_test_samples = preview_generations(
    class_id=3,
    n_samples=5,
    start_index=200,
    show_demonstrations=True,
)

Target class: Sexism
Demonstration source: Random authentic same-class RUHSOLD examples

Sample 1
------------------------------------------------------------
Sample index: 200
Generation seed: 301200
Demonstration seed: 305200

Demonstrations:
1. aur yeah hijra #billorani kuch nahin ban sakta. 🤮 baray moun wali looti. 🙄 #bilawajabhutto #accidentalchairman #fakebhutto #abubachao
2. bata ab aurat ke naam ke kalakh
3. this randi should also tell thst the babri masjid was built on demolished temple
4. chhee kitni gandi shakal 🤮🤮🤮🤣🤣😂 bhakkk randi saali jisko manti hai usse to dar

Generated output:
nhi to acha hoga..😂

Sample 2
------------------------------------------------------------
Sample index: 201
Generation seed: 301201
Demonstration seed: 305201

Demonstrations:
1. #rashamidesai tu randi moti apni maa ko bech chuki hai ...kisi aur ke hatho..to apni awkat ne rah kar game plan kar ...arhan to vaduaa hai...ram mandir ban rha hai na..but tu chud rahi hai kisi aur ka bister garam kar 

In [21]:
# ============================================================
# Display ALL original RUHSOLD Sexism samples for review
# ============================================================

sexism_candidates = (
    demonstration_pool[
        demonstration_pool["label"] == 3
    ][["text", "label"]]
    .dropna(subset=["text"])
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

# Add stable IDs for manual review and later selection
sexism_candidates.insert(
    0,
    "review_id",
    range(len(sexism_candidates))
)

# Display everything without truncating the text
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

print(
    "Total unique Sexism candidates:",
    len(sexism_candidates)
)

#display(sexism_candidates)

Total unique Sexism candidates: 535


In [22]:
# ============================================================
# SEXISM PROTOTYPE KEEP IDS
# ============================================================

SEXISM_PROTOTYPE_KEEP_IDS = [
    # Gender-role / women stereotypes
    18,
    67,
    70,
    92,
    95,
    124,
    153,
    161,
    179,
    190,
    222,
    241,
    269,
    271,
    290,
    310,
    324,
    359,
    432,
    459,
    466,
    470,
    476,
    514,

    # Gender identity / masculinity degradation
    14,
    78,
    123,
    217,
    221,
    331,
    337,
    367,
    414,
    417,
    478,
    509,

    # Explicit sexual / gendered degradation
    49,
    150,
    258,
    462,
]

In [23]:
# ============================================================
# CREATE SEXISM PROTOTYPE POOL
# ============================================================

sexism_prototype_pool = (
    sexism_candidates[
        sexism_candidates["review_id"].isin(
            SEXISM_PROTOTYPE_KEEP_IDS
        )
    ][
        [
            "text",
            "label",
        ]
    ]
    .drop_duplicates(
        subset=["text"]
    )
    .reset_index(drop=True)
)

print(
    "Sexism prototype candidates:",
    len(sexism_prototype_pool)
)

display(
    sexism_prototype_pool
)

Sexism prototype candidates: 40


,text,label
0,70 saal ka mard is better than a 30 year old hijra😅,3
1,jin ko hizab se nafrat he wo nangi ho k ghoome koi in pe thooke ga bhi nahi aurat khoobsurat pure libas me lagti he mawazna karlen,3
2,aisi or kis kis ki lund par uchel uchel kar moja lati ghar me.,3
3,#stopauratbarbadimarch aurat ghr k andr hi achi lgti he yu sarkon par nikalna kisi bhi aurat ki izzat par daag lga skta he,3
4,randi kise kahte ha aap hi bata do? ye word ghatiya women ke liye hi use karta ha ab deepika ghatiya ha to randi hi bolenge na,3
5,phir larky beard ky bagiar khusray q lgty hain wo bhi beard ky bagiar khubsurat lgny chahiye ?,3
6,original attitude boys ma hota ha aur fake girls ma,3
7,mard muhbt kr bhi ly tu aksar aurat he dhoka dy jati h usy 💔,3
8,ap ko mard banny ki acting pasand i? agr ni to hijra hi bany rahye sir ye mardo k kam hn hijro k ni.,3
9,acha tareka dhondha hai larkun ko ganda sabit karny ka but aesi neech harkat larkyan he kar sakti hain larky nahi.,3


In [99]:
sexism_test_samples = preview_generations(
    class_id=3,
    n_samples=5,
    start_index=200,
    show_demonstrations=True,
)

Target class: Sexism
Demonstration source: Shortlisted authentic RUHSOLD Sexism examples
Demonstration pool size: 40

Sample 1
------------------------------------------------------------
Sample index: 200
Generation seed: 301200
Demonstration seed: 305200

Demonstrations:
1. original attitude boys ma hota ha aur fake girls ma
2. your leader  is hijra that's why he is not living with his wife.
3. yeh wesy koi bht e zyda ounchi nasal ki randi niqli h wesy , mtlb i can't even imagine koi bughaz or paisy k pechay itna b girr skta h, nehayat e nichly darjay ki baighairat aurat ho, aurat zaat ko b badnaam kea h isny apny wajood sy !! akhh thooo iski zaat pay
4. agar aaj se 40 saal pehley isi tarha ke banner pakarh kar mera jism meri marzi waala naara aapki amma ne maara hota to aap is duniya main ye tweet karney ke liye maujood hi na hoteen 😑😑😑

Generated output:
isky ghar mein toh pata nahi chal raha kis ny ungli kr di

Sample 2
------------------------------------------------------------


In [100]:
# ============================================================
# INITIAL PROFANE SANITY CHECK
# 5 samples using original RUHSOLD demonstrations
# ============================================================

profane_test_samples = preview_generations(
    class_id=4,
    n_samples=5,
    start_index=300,
    show_demonstrations=True,
)

Target class: Profane
Demonstration source: Shortlisted authentic RUHSOLD Profane examples
Demonstration pool size: 40

Sample 1
------------------------------------------------------------
Sample index: 300
Generation seed: 401300
Demonstration seed: 405300

Demonstrations:
1. rt : asa kon ludo khelta hi bc ....😇
2. mere pas tum ho ny to puray saal ka rula dia bhenchod
3. tm jao ma ni ja rha, pehly he adhi zindagi saffr mein guzr gai bc meri.
4. bhenchod bht kutti haaalat huwe we he bhai

Generated output:
mera baap chutiya hai

Sample 2
------------------------------------------------------------
Sample index: 301
Generation seed: 401301
Demonstration seed: 405301

Demonstrations:
1. bhenchod mairi thand se phatt rahi hai
2. rona araha hai bhenchod why am i such a clown why am i so unimportant
3. why can’t i multitask???? i’m 22 fucking years old and i still dont know how to multitask bhenchod
4. rt : koi physics samjha de bhenchod,,,,

Generated output:
yeh kya lagta hai? abhi tak m

In [24]:
# ============================================================
# PREPARE ORIGINAL PROFANE CANDIDATES
# ============================================================

profane_candidates = (
    train_df[
        train_df["label"] == 4
    ][["text", "label"]]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print(
    "Total unique Profane candidates:",
    len(profane_candidates)
)

Total unique Profane candidates: 410


In [25]:
# ============================================================
# BUILD SHORTLISTED PROFANE PROTOTYPE POOL
# ============================================================

PROFANE_PROTOTYPE_INDICES = [
    2, 3, 4, 5, 13, 28, 30, 32, 36, 38,
    39, 42, 48, 49, 53, 54, 60, 62, 68, 69,
    73, 74, 76, 77, 79, 85, 90, 91, 93, 96,
    98, 103, 104, 112, 119, 130, 137, 143,
    152, 153,
]

profane_prototype_pool = (
    profane_candidates
    .loc[
        PROFANE_PROTOTYPE_INDICES,
        ["text", "label"]
    ]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print(
    "Profane prototype candidates:",
    len(profane_prototype_pool)
)

#display(profane_prototype_pool)

Profane prototype candidates: 40


In [26]:
# ============================================================
# PROFANE SANITY CHECK
# 5 samples using shortlisted prototype pool
# ============================================================

profane_test_samples = preview_generations(
    class_id=4,
    n_samples=5,
    start_index=300,
    show_demonstrations=True,
)

Target class: Profane
Demonstration source: Shortlisted authentic RUHSOLD Profane examples
Demonstration pool size: 40

Sample 1
------------------------------------------------------------
Sample index: 300
Generation seed: 401300
Demonstration seed: 405300

Demonstrations:
1. rt : asa kon ludo khelta hi bc ....😇
2. mere pas tum ho ny to puray saal ka rula dia bhenchod
3. tm jao ma ni ja rha, pehly he adhi zindagi saffr mein guzr gai bc meri.
4. bhenchod bht kutti haaalat huwe we he bhai

Generated output:
mera baap chutiya hai

Sample 2
------------------------------------------------------------
Sample index: 301
Generation seed: 401301
Demonstration seed: 405301

Demonstrations:
1. bhenchod mairi thand se phatt rahi hai
2. rona araha hai bhenchod why am i such a clown why am i so unimportant
3. why can’t i multitask???? i’m 22 fucking years old and i still dont know how to multitask bhenchod
4. rt : koi physics samjha de bhenchod,,,,

Generated output:
yeh kya lagta hai? abhi tak m

In [27]:
# ============================================================
# PROJECT PATH SETUP
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nExists:")
print(PROJECT_ROOT.exists())

Project root:
/home/jovyan/project work/data_analyssis

Exists:
True


In [107]:
# ============================================================
# GENERATE 100 PROFANE PILOT SAMPLES
# ============================================================

profane_pilot_df = generate_pilot_dataset(
    class_id=4,
    n_samples=100,
    start_index=0,
    save_every=10,
)

Generating pilot dataset
Class: Profane
Demonstration source: Shortlisted authentic RUHSOLD Profane examples
Demonstration pool size: 40
Samples: 100
Output: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/profane_raw_pilot_100.csv
Saved 10/100
Saved 20/100
Saved 30/100
Saved 40/100
Saved 50/100
Saved 60/100
Saved 70/100
Saved 80/100
Saved 90/100
Saved 100/100

Generation complete.
Shape: (100, 14)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/profane_raw_pilot_100.csv


In [108]:
# ============================================================
# GENERATE 100 SEXISM PILOT SAMPLES
# ============================================================

sexism_pilot_df = generate_pilot_dataset(
    class_id=3,
    n_samples=100,
    start_index=0,
    save_every=10,
)

Generating pilot dataset
Class: Sexism
Demonstration source: Shortlisted authentic RUHSOLD Sexism examples
Demonstration pool size: 40
Samples: 100
Output: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/sexism_raw_pilot_100.csv
Saved 10/100
Saved 20/100
Saved 30/100
Saved 40/100
Saved 50/100
Saved 60/100
Saved 70/100
Saved 80/100
Saved 90/100
Saved 100/100

Generation complete.
Shape: (100, 14)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/pilot/sexism_raw_pilot_100.csv


In [29]:
# ============================================================
# GENERATE ONE FINAL PRODUCTION BATCH
# ROUND-2 SAFE VERSION
# ============================================================

from pathlib import Path
import pandas as pd


def generate_production_batch(
    class_id,
    batch_number,
    start_index,
    batch_size=200,
    output_dir=None,
):
    """
    Generate one reproducible production batch
    for one RUHSOLD augmentation class.

    Supported classes:
        2 = Religious Hate
        3 = Sexism
        4 = Profane

    Parameters
    ----------
    class_id : int
        RUHSOLD target class ID.

    batch_number : int
        Batch number used in filenames and candidate IDs.

    start_index : int
        Starting sample index for deterministic seeds.

    batch_size : int
        Number of samples to generate.

    output_dir : Path or str, optional
        Directory where the generated batch will be saved.
        If None, FULL_OUTPUT_DIR is used.
    """

    # --------------------------------------------------------
    # Validate class
    # --------------------------------------------------------

    if class_id not in [2, 3, 4]:
        raise ValueError(
            "Production generation is only configured "
            "for classes 2, 3, and 4."
        )

    # --------------------------------------------------------
    # Select output directory
    # --------------------------------------------------------

    if output_dir is None:
        output_dir = FULL_OUTPUT_DIR

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Select curated demonstration pool
    # --------------------------------------------------------

    if class_id == 2:

        demo_source_df = religious_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Religious Hate examples"
        )

    elif class_id == 3:

        demo_source_df = sexism_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Sexism examples"
        )

    elif class_id == 4:

        demo_source_df = profane_prototype_pool

        demo_source_name = (
            "Shortlisted authentic RUHSOLD Profane examples"
        )

    # --------------------------------------------------------
    # Build class-specific filename
    # --------------------------------------------------------

    class_file_name = (
        CLASS_METADATA[class_id]["label"]
        .lower()
        .replace("/", "_")
        .replace(" ", "_")
    )

    output_path = (
        output_dir
        / (
            f"{class_file_name}"
            f"_batch_{batch_number:03d}.csv"
        )
    )

    # --------------------------------------------------------
    # Prevent accidental overwrite
    # --------------------------------------------------------

    if output_path.exists():

        raise FileExistsError(
            f"Output file already exists:\n{output_path}\n\n"
            "Choose a new batch number or delete the file "
            "only if you intentionally want to regenerate it."
        )

    # --------------------------------------------------------
    # Display batch configuration
    # --------------------------------------------------------

    print("=" * 80)
    print("PRODUCTION GENERATION")
    print("=" * 80)

    print(
        "Class:",
        CLASS_METADATA[class_id]["label"]
    )

    print(
        "Batch number:",
        batch_number
    )

    print(
        "Batch size:",
        batch_size
    )

    print(
        "Start index:",
        start_index
    )

    print(
        "Demonstration source:",
        demo_source_name
    )

    print(
        "Demonstration pool size:",
        len(demo_source_df)
    )

    print(
        "Output directory:",
        output_dir
    )

    print(
        "Output file:",
        output_path
    )

    print("=" * 80)

    # --------------------------------------------------------
    # Generate batch
    # --------------------------------------------------------

    records = []

    for offset in range(
        batch_size
    ):

        sample_index = (
            start_index
            + offset
        )

        # ----------------------------------------------------
        # Deterministic generation seeds
        # ----------------------------------------------------

        generation_seed = (
            BASE_GENERATION_SEED
            + class_id * 100000
            + sample_index
        )

        demonstration_seed = (
            BASE_DEMONSTRATION_SEED
            + class_id * 100000
            + sample_index
        )

        # ----------------------------------------------------
        # Sample demonstrations
        # ----------------------------------------------------

        demonstrations = (
            sample_demonstrations(
                dataframe=demo_source_df,
                class_id=class_id,
                n_examples=N_DEMONSTRATIONS,
                random_state=demonstration_seed,
            )
        )

        # ----------------------------------------------------
        # Build few-shot prompt
        # ----------------------------------------------------

        messages = (
            build_few_shot_messages(
                class_id=class_id,
                demonstrations=demonstrations,
            )
        )

        # ----------------------------------------------------
        # Generate synthetic sample
        # ----------------------------------------------------

        generated_text = (
            generate_one(
                model=finetuned_model,
                messages=messages,
                seed=generation_seed,
            )
        )

        # ----------------------------------------------------
        # Unique production candidate ID
        # ----------------------------------------------------

        production_candidate_id = (
            f"class{class_id}_"
            f"batch{batch_number:03d}_"
            f"sample{sample_index:06d}"
        )

        # ----------------------------------------------------
        # Store generation record
        # ----------------------------------------------------

        records.append({
            "production_candidate_id":
                production_candidate_id,

            "batch_number":
                batch_number,

            "sample_index":
                sample_index,

            "class_id":
                class_id,

            "target_label":
                CLASS_METADATA[
                    class_id
                ]["label"],

            "generation_seed":
                generation_seed,

            "demonstration_seed":
                demonstration_seed,

            "prompt_version":
                PROMPT_VERSION,

            "prompting_strategy":
                "4-shot random authentic same-class",

            "demonstration_source":
                demo_source_name,

            "demonstration_pool_size":
                len(demo_source_df),

            "demo_1":
                demonstrations[0],

            "demo_2":
                demonstrations[1],

            "demo_3":
                demonstrations[2],

            "demo_4":
                demonstrations[3],

            "generated_text":
                generated_text,
        })

        # ----------------------------------------------------
        # Incremental save every 25 samples
        # ----------------------------------------------------

        if (
            (offset + 1) % 25 == 0
            or
            (offset + 1) == batch_size
        ):

            temporary_df = (
                pd.DataFrame(
                    records
                )
            )

            temporary_df.to_csv(
                output_path,
                index=False,
                encoding="utf-8"
            )

            print(
                f"Saved "
                f"{offset + 1}/{batch_size}"
            )

    # --------------------------------------------------------
    # Final dataframe
    # --------------------------------------------------------

    batch_df = (
        pd.DataFrame(
            records
        )
    )

    # --------------------------------------------------------
    # Final save
    # --------------------------------------------------------

    batch_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    print("\nBatch generation complete.")

    print(
        "Shape:",
        batch_df.shape
    )

    print(
        "Saved to:",
        output_path
    )

    return batch_df

In [30]:
# ============================================================
# ROUND 2 OUTPUT DIRECTORY
# ============================================================

ROUND2_OUTPUT_DIR = (
    GENERATION_DIR
    / "full_round2"
)

ROUND2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Round 2 output directory:"
)

print(
    ROUND2_OUTPUT_DIR
)

Round 2 output directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2


In [46]:
# ============================================================
# RELIGIOUS HATE - ROUND 2
# Target: 900 additional raw generations
#
# Existing production:
# batches 1-5
# sample indices 10000-10999
#
# New production:
# batches 6-10
# sample indices 11000-11899
# ============================================================

# 4 batches x 200 = 800 samples
for batch_number in range(6, 10):

    start_index = (
        11000
        + (batch_number - 6) * 200
    )

    generate_production_batch(
        class_id=2,
        batch_number=batch_number,
        start_index=start_index,
        batch_size=200,
        output_dir=ROUND2_OUTPUT_DIR,
    )

# Final 100 samples
generate_production_batch(
    class_id=2,
    batch_number=10,
    start_index=11800,
    batch_size=100,
    output_dir=ROUND2_OUTPUT_DIR,
)

PRODUCTION GENERATION
Class: Religious Hate
Batch number: 6
Batch size: 200
Start index: 11000
Demonstration source: Shortlisted authentic RUHSOLD Religious Hate examples
Demonstration pool size: 118
Output directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2
Output file: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/religious_hate_batch_006.csv
Saved 25/200
Saved 50/200
Saved 75/200
Saved 100/200
Saved 125/200
Saved 150/200
Saved 175/200
Saved 200/200

Batch generation complete.
Shape: (200, 16)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/religious_hate_batch_006.csv
PRODUCTION GENERATION
Class: Religious Hate
Batch number: 7
Batch size: 200
Start index: 11200
Demonstration source: Shortlisted authentic RUHSOLD Religious Hate examples
Demonstration pool size: 118
Output directory: /home/jovyan/project work/data_analyssis/fine t

,production_candidate_id,batch_number,sample_index,class_id,target_label,generation_seed,demonstration_seed,prompt_version,prompting_strategy,demonstration_source,demonstration_pool_size,demo_1,demo_2,demo_3,demo_4,generated_text
0,class2_batch010_sample011800,10,11800,2,Religious Hate,212800,216800,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Religious Hate examples,118,apne profile mein whats app me fb twitter mein is bahadur nojawaan ki pic lga kr islam se or nabi ke din se mhbt ka izhaar kren or itna share kren ek yahoodi apne na pak iradoon se baaz ajaye is shakhs ne quraan pak ki hifazahat mein ek bhtareen mhbt ka jita jagta sabot dia hai,o madadarchod: tum tu farigh ho aur yeh gashti tu waisay he sari raat jagti hay. o dallay o kanjar hum tu iran jatay hotay hain tu bata apni behan kitni baar mbs aur us kay wahabi sheikhon ko bhaijta hay.,abe mc mein bheju teri maa cow ka photo kat te hue bc. chod teri tarah andhbhakt nhi hu aur nhi illiterate.. go and drink cow piss.,yahoodi ko hitler ne chora or general ne in jaajoon ko chor k dhkya k ye kai hain.😂,tere ghar ka gharaq karwa diya tha jo tere ammi ko mara tha.
1,class2_batch010_sample011801,10,11801,2,Religious Hate,212801,216801,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Religious Hate examples,118,"rt : q. woh kaun si qome he,jis ne such bhi bola phr bhi jahnum mein jayegi? ans: yahoodi or nasara qome. shahid sindh",wow halala's product on fire as usual porkistaniðÿ˜‚ðÿ˜‚ðÿ˜‚,qadiyani pe lanat mirza khinzeer harami pe lanat.,get lost you kaafir.,abe teri maa chodta ha apni nikal gaya
2,class2_batch010_sample011802,10,11802,2,Religious Hate,212802,216802,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Religious Hate examples,118,ahmedi mirziee qadiyani hargiz kuleedi uhdu pe nhee janay chahien warna hmari agencies pe swalia nishan hey,jaa bhen k lory tum harrami ki nasal ho.. dewband maslak pe kutty ki tareh bhoonkty ho..,apne profile mein whats app me fb twitter mein is bahadur nojawaan ki pic lga kr islam se or nabi ke din se mhbt ka izhaar kren or itna share kren ek yahoodi apne na pak iradoon se baaz ajaye is shakhs ne quraan pak ki hifazahat mein ek bhtareen mhbt ka jita jagta sabot dia hai,"rt : q. woh kaun si qome he,jis ne such bhi bola phr bhi jahnum mein jayegi? ans: yahoodi or nasara qome. shahid sindh",qadyani kanjar aurat
3,class2_batch010_sample011803,10,11803,2,Religious Hate,212803,216803,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Religious Hate examples,118,abe bharwe tum hindu hi duniya k sub se bara atankwaz ho haramiyon bt krte ho hamare age tumhari country me hindu k elawa koi safe nhi he tumhari country ka name isi liye lynchistan he isi king all the world,uff pakiza muslim randi 😉😍,oeyy randdii k bachay ganddu hindu.. kyun jhoott baq raha hai gashti maa k ganndduu bachay. peshaab p k palnay wali qoum.,in beghairto israel yahodi khanzeer kai bacho nai kia kuch nahen kia hain lekhin alhamdulillah islam wahid deen jo mohabbat sikhati hain nakai nafrat aur ye sab sai tezi sai phelta huwa deen hain is liye inko jalan ho rahi hain aur har koshish kar bhe chukai hain aur kar bhe,tum logon ne apni amma ko khud kah diya tha ke tera baap hindu hai aur tu poore hindustan ki haramzadi beti hai
4,class2_batch010_sample011804,10,11804,2,Religious Hate,212804,216804,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Religious Hate examples,118,chhor bhen chod hrami hindu chhor esko😠😠😠,sali muslims randi. jihadi sali,plz sir hamy in say koi khtra nahi hamy is imran khan yahodi our kadiyanion say bachny pay gor karen pak army tayri azmat ko salam,"bilkul,itny lecture milny hain jese christmas or new year pe milty hain ,kya apko pata bhi hai k kaafir janwaro k haqq main dua bhi nahi krni chahye",oye kutiya ye

In [47]:
# ============================================================
# SEXISM - ROUND 2
# Target: 1300 additional raw generations
#
# Existing production:
# batches 1-4
# sample indices 20000-20799
#
# New production:
# batches 5-11
# sample indices 20800-22099
# ============================================================

# 6 batches x 200 = 1200 samples
for batch_number in range(5, 11):

    start_index = (
        20800
        + (batch_number - 5) * 200
    )

    generate_production_batch(
        class_id=3,
        batch_number=batch_number,
        start_index=start_index,
        batch_size=200,
        output_dir=ROUND2_OUTPUT_DIR,
    )

# Final 100 samples
generate_production_batch(
    class_id=3,
    batch_number=11,
    start_index=22000,
    batch_size=100,
    output_dir=ROUND2_OUTPUT_DIR,
)

PRODUCTION GENERATION
Class: Sexism
Batch number: 5
Batch size: 200
Start index: 20800
Demonstration source: Shortlisted authentic RUHSOLD Sexism examples
Demonstration pool size: 40
Output directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2
Output file: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/sexism_batch_005.csv
Saved 25/200
Saved 50/200
Saved 75/200
Saved 100/200
Saved 125/200
Saved 150/200
Saved 175/200
Saved 200/200

Batch generation complete.
Shape: (200, 16)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/sexism_batch_005.csv
PRODUCTION GENERATION
Class: Sexism
Batch number: 6
Batch size: 200
Start index: 21000
Demonstration source: Shortlisted authentic RUHSOLD Sexism examples
Demonstration pool size: 40
Output directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2
Out

,production_candidate_id,batch_number,sample_index,class_id,target_label,generation_seed,demonstration_seed,prompt_version,prompting_strategy,demonstration_source,demonstration_pool_size,demo_1,demo_2,demo_3,demo_4,generated_text
0,class3_batch011_sample022000,11,22000,3,Sexism,323000,327000,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,ap ko mard banny ki acting pasand i? agr ni to hijra hi bany rahye sir ye mardo k kam hn hijro k ni.,tu kyun ro rahe hai hijra? tera buddhe hijra to kamjor mard hai. uska kara nehi hota. 🤣🤣🤣🤣,#stopauratbarbadimarch aurat ghr k andr hi achi lgti he yu sarkon par nikalna kisi bhi aurat ki izzat par daag lga skta he,transgander madarchood hijra tu aik bara khoota ha haram zada,tu jis ny maulana ko bola tha wo teri maa behno ka phattar hoga...tujhy bhi aisa samjha hoga..
1,class3_batch011_sample022001,11,22001,3,Sexism,323001,327001,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,ek randi larki ko itni pazeerai di hai. smjh se bahir hai,randi kise kahte ha aap hi bata do? ye word ghatiya women ke liye hi use karta ha ab deepika ghatiya ha to randi hi bolenge na,r mazy k bat mra jism wo auratn krti hn jin pr thokta bh koi nae.,agar aaj se 40 saal pehley isi tarha ke banner pakarh kar mera jism meri marzi waala naara aapki amma ne maara hota to aap is duniya main ye tweet karney ke liye maujood hi na hoteen 😑😑😑,apke baare me sari dunia sharm khud rahi hai
2,class3_batch011_sample022002,11,22002,3,Sexism,323002,327002,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,#auratmarch2020 mera jism allah ki marzi.. allah ka to hukam ha tum char dewari me raho .. nd ya nikl k apni nimaish karna chahten,randi kise kahte ha aap hi bata do? ye word ghatiya women ke liye hi use karta ha ab deepika ghatiya ha to randi hi bolenge na,#stopauratbarbadimarch aurat ghr k andr hi achi lgti he yu sarkon par nikalna kisi bhi aurat ki izzat par daag lga skta he,ap ko mard banny ki acting pasand i? agr ni to hijra hi bany rahye sir ye mardo k kam hn hijro k ni.,teri maa chod di thi tu samjh le
3,class3_batch011_sample022003,11,22003,3,Sexism,323003,327003,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,#auratmarch2020 mera jism allah ki marzi.. allah ka to hukam ha tum char dewari me raho .. nd ya nikl k apni nimaish karna chahten,dhoka khane k baad oska aitbar uth jata haa sbsee or dosre mard mai b interest ni rehta...ku k aurat jism k ley ni chahti jbk mard jism k ley chahta😌🙂,tum jesi ghatia aurat ka kam hi ni hai media par.kisi kothhay pe ja aur ulta late ja ke.begherat,transgander madarchood hijra tu aik bara khoota ha haram zada,yar teri maa ko dekhna hoga na bsdk
4,class3_batch011_sample022004,11,22004,3,Sexism,323004,327004,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,we are supporting #islam and thank you #khalilurrahmanqamar for your beautiful comments against #gashti . agar ye #aurat hoti to na me aesi zuban use krta aur na koi gairatmand #admi use krta.,ek randi larki ko itni pazeerai di hai. smjh se bahir hai,muslim countries ki leaderships na mard napunsak or hijra hain.. in mein koi dum nahi.. ye kuch nahi kr sakty.. jinnah ki rooh tarapti hogi.. jinnah ne kashmir par attack ka order diya tha..,teri bhosdi or boobs me parda mat kar...logo ko dekne de.dhake hoge to kaise pata chalega ki tu hijra nehi hai. hijra kahi ki.,kisi randi ko pta chalna chaheye hai uski maa behn ka naam
5,class3_batch011_sample022005,11,22005,3,Sexism,323005,327005,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Sexism examples,40,teri bhosdi or boobs me parda mat kar...logo ko dekne de.dhake hoge to kaise pata chalega ki tu hijra nehi hai. hij

In [48]:
# ============================================================
# PROFANE - ROUND 2
# Target: 450 additional raw generations
#
# Existing production:
# batches 1-3
# sample indices 30000-30599
#
# New production:
# batches 4-6
# sample indices 30600-31049
# ============================================================

# 2 batches x 200 = 400 samples
for batch_number in range(4, 6):

    start_index = (
        30600
        + (batch_number - 4) * 200
    )

    generate_production_batch(
        class_id=4,
        batch_number=batch_number,
        start_index=start_index,
        batch_size=200,
        output_dir=ROUND2_OUTPUT_DIR,
    )

# Final 50 samples
generate_production_batch(
    class_id=4,
    batch_number=6,
    start_index=31000,
    batch_size=50,
    output_dir=ROUND2_OUTPUT_DIR,
)

PRODUCTION GENERATION
Class: Profane
Batch number: 4
Batch size: 200
Start index: 30600
Demonstration source: Shortlisted authentic RUHSOLD Profane examples
Demonstration pool size: 40
Output directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2
Output file: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/profane_batch_004.csv
Saved 25/200
Saved 50/200
Saved 75/200
Saved 100/200
Saved 125/200
Saved 150/200
Saved 175/200
Saved 200/200

Batch generation complete.
Shape: (200, 16)
Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2/profane_batch_004.csv
PRODUCTION GENERATION
Class: Profane
Batch number: 5
Batch size: 200
Start index: 30800
Demonstration source: Shortlisted authentic RUHSOLD Profane examples
Demonstration pool size: 40
Output directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_roun

,production_candidate_id,batch_number,sample_index,class_id,target_label,generation_seed,demonstration_seed,prompt_version,prompting_strategy,demonstration_source,demonstration_pool_size,demo_1,demo_2,demo_3,demo_4,generated_text
0,class4_batch006_sample031000,6,31000,4,Profane,432000,436000,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,bhenchod mairi thand se phatt rahi hai,mera bhi yaar bhai...jindgi jhand ho gya hai bhenchodv 3 t20 me 300 rupya haar gye....,inn coaching vaalou ko kaise pta chal jaata hai mera result bhenchod.,naak bandd ho gya bc.,aby teri maa behn ka number dekh lete to acha nahi lagta
1,class4_batch006_sample031001,6,31001,4,Profane,432001,436001,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,my entire neighborhood is making weird sounds bhenchod shaddi hai kya,kyaaaa musibbaattt ha yr sara mood khraab kr diya ha bc,i hate ors bhenchod,bhenchod parhna hai. par dimaagh ijazat nahi deta. ajeeb chutiyapa hai ye.,bhenchod why do we even care about such things?
2,class4_batch006_sample031002,6,31002,4,Profane,432002,436002,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,rona araha hai bhenchod why am i such a clown why am i so unimportant,inn coaching vaalou ko kaise pta chal jaata hai mera result bhenchod.,bhenchod what a goal!,rt : asa kon ludo khelta hi bc ....😇,ye bhenchod
3,class4_batch006_sample031003,6,31003,4,Profane,432003,436003,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,dil sumbhal ja zra bhenchod,bhenchod bht kutti haaalat huwe we he bhai,rt : birthday wali feel hi nai arahi bhenchod,bhenchod kya hai ye sb,bc hn mein ne bhi bola tha maine
4,class4_batch006_sample031004,6,31004,4,Profane,432004,436004,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,bhenchod kya hai ye sb,ajeeeeb bc 😒😂,main 5'8 ke sath bh chota lgta hn bc😶😑😑😑,bhenchod syllabus kuch aur..aur padhke kuch aur gaya.,bhenchod yeh sahi kia hai
5,class4_batch006_sample031005,6,31005,4,Profane,432005,436005,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,rt : asa kon ludo khelta hi bc ....😇,bhenchod aaj sirf roneka mann kar raha hai,kyaaaa musibbaattt ha yr sara mood khraab kr diya ha bc,"tm jao ma ni ja rha, pehly he adhi zindagi saffr mein guzr gai bc meri.",😂😂😂😂😂😂 ye chutiye ajeeb se pata nhi kis ny likha tha
6,class4_batch006_sample031006,6,31006,4,Profane,432006,436006,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,bhenchod bht kutti haaalat huwe we he bhai,bhenchod aaj sirf roneka mann kar raha hai,why can’t i multitask???? i’m 22 fucking years old and i still dont know how to multitask bhenchod,"tm jao ma ni ja rha, pehly he adhi zindagi saffr mein guzr gai bc meri.",kbhi bhi bhenchod nahi bolta kabhi
7,class4_batch006_sample031007,6,31007,4,Profane,432007,436007,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,bhenchod what a goal!,bhenchod kya hai ye sb,my entire neighborhood is making weird sounds bhenchod shaddi hai kya,code bhee saaf karna hai abheeee bc,mc chup krta hu aurat march ki baat kar raha hu😂
8,class4_batch006_sample031008,6,31008,4,Profane,432008,436008,v3_original_demo_boundary_prompt,4-shot random authentic same-class,Shortlisted authentic RUHSOLD Profane examples,40,dil sumbhal ja zra bhenchod,bhenchod aaj sirf roneka mann kar raha hai,mera bhi yaar bhai...jindgi jhand ho gya hai bhenchodv 3 t20 me 300 rupya haar gye....,bhenchod syllabus kuch aur..aur padhke kuch aur gaya.,rt : itna kia pta chl rha ha tu bhenchod
9,class4_batch006_sample031009,6,31009,4,Prof

In [49]:
# ============================================================
# VERIFY ROUND 2 GENERATION FILES
# ============================================================

from pathlib import Path
import pandas as pd

ROUND2_OUTPUT_DIR = (
    GENERATION_DIR
    / "full_round2"
)

round2_files = sorted(
    ROUND2_OUTPUT_DIR.glob("*.csv")
)

print(
    "Round 2 files found:",
    len(round2_files)
)

for path in round2_files:
    df = pd.read_csv(path)

    print(
        path.name,
        "->",
        len(df)
    )

Round 2 files found: 15
profane_batch_004.csv -> 200
profane_batch_005.csv -> 200
profane_batch_006.csv -> 50
religious_hate_batch_006.csv -> 200
religious_hate_batch_007.csv -> 200
religious_hate_batch_008.csv -> 200
religious_hate_batch_009.csv -> 200
religious_hate_batch_010.csv -> 100
sexism_batch_005.csv -> 200
sexism_batch_006.csv -> 200
sexism_batch_007.csv -> 200
sexism_batch_008.csv -> 200
sexism_batch_009.csv -> 200
sexism_batch_010.csv -> 200
sexism_batch_011.csv -> 100


In [50]:
# ============================================================
# COMBINE ROUND 2 RAW GENERATIONS
# ============================================================

round2_frames = []

for path in round2_files:

    df = pd.read_csv(path)

    df["source_batch_file"] = (
        path.name
    )

    round2_frames.append(
        df
    )


round2_raw_df = pd.concat(
    round2_frames,
    ignore_index=True,
)


print(
    "Combined Round 2 shape:",
    round2_raw_df.shape
)

print(
    "\nClass distribution:"
)

display(
    round2_raw_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)


print(
    "\nDuplicate candidate IDs:",
    round2_raw_df[
        "production_candidate_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Missing generated texts:",
    round2_raw_df[
        "generated_text"
    ]
    .isna()
    .sum()
)

print(
    "Exact duplicate generated texts:",
    round2_raw_df[
        "generated_text"
    ]
    .duplicated()
    .sum()
)

Combined Round 2 shape: (2650, 17)

Class distribution:


,Class,Count
0,Sexism,1300
1,Religious Hate,900
2,Profane,450



Duplicate candidate IDs: 0
Missing generated texts: 0
Exact duplicate generated texts: 30
